# Pipeline Observability Metrics

Heavy notebook for phase-1 observability.

Responsibilities:

- summarize current pipeline freshness into `obs.obs_ingestion_state`
- aggregate Silver quarantine counts into `obs.obs_dq_metrics`
- write an execution record for this notebook into `obs.obs_pipeline_run_log`

This notebook does not backfill historical pipeline run events for Bronze/Silver/Gold tasks. It only observes the current table state and quarantine metrics.

In [ ]:
import json
import uuid
from datetime import datetime, timezone
from functools import reduce

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType

UTC = timezone.utc

RUN_LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("source_name", StringType(), True),
    StructField("target_table", StringType(), False),
    StructField("status", StringType(), False),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("started_at", TimestampType(), False),
    StructField("completed_at", TimestampType(), True),
    StructField("error_message", StringType(), True),
    StructField("metadata_json", StringType(), True),
])

INGESTION_STATE_SCHEMA = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("source_name", StringType(), True),
    StructField("target_table", StringType(), False),
    StructField("watermark_value", StringType(), True),
    StructField("watermark_type", StringType(), True),
    StructField("last_success_at", TimestampType(), True),
    StructField("last_run_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("updated_at", TimestampType(), False),
])


def build_monitored_pipelines(catalog: str):
    return [
        {
            "pipeline_name": "bronze_market_coinbase",
            "source_name": "coinbase",
            "target_table": f"{catalog}.brz_market.raw_coinbase_ohlc_1d",
            "watermark_column": "bar_date",
            "watermark_type": "bar_date",
            "filter_sql": None,
        },
        {
            "pipeline_name": "bronze_macro_ecb_fx_ref_rates_daily",
            "source_name": "ecb",
            "target_table": f"{catalog}.brz_macro.raw_ecb_fx_ref_rates_daily",
            "watermark_column": "rate_date",
            "watermark_type": "rate_date",
            "filter_sql": None,
        },
        {
            "pipeline_name": "bronze_macro_fred_series",
            "source_name": "fred",
            "target_table": f"{catalog}.brz_macro.raw_fred_series",
            "watermark_column": "observation_date",
            "watermark_type": "observation_date",
            "filter_sql": None,
        },
        {
            "pipeline_name": "silver_market_crypto_ohlc_1d",
            "source_name": "coinbase",
            "target_table": f"{catalog}.slv_market.crypto_ohlc_1d",
            "watermark_column": "bar_date",
            "watermark_type": "bar_date",
            "filter_sql": None,
        },
        {
            "pipeline_name": "silver_macro_ecb_fx_ref_rates_daily",
            "source_name": "ecb",
            "target_table": f"{catalog}.slv_macro.ecb_fx_ref_rates_daily",
            "watermark_column": "rate_date",
            "watermark_type": "rate_date",
            "filter_sql": None,
        },
        {
            "pipeline_name": "silver_macro_fred_series_clean",
            "source_name": "fred",
            "target_table": f"{catalog}.slv_macro.fred_series_clean",
            "watermark_column": "observation_date",
            "watermark_type": "observation_date",
            "filter_sql": None,
        },
        {
            "pipeline_name": "gold_market_crypto_returns_1d",
            "source_name": "coinbase",
            "target_table": f"{catalog}.gld_market.dp_crypto_returns_1d",
            "watermark_column": "bar_date",
            "watermark_type": "bar_date",
            "filter_sql": None,
        },
        {
            "pipeline_name": "gold_market_crypto_volatility_1d",
            "source_name": "coinbase",
            "target_table": f"{catalog}.gld_market.dp_crypto_volatility_1d",
            "watermark_column": "bar_date",
            "watermark_type": "bar_date",
            "filter_sql": None,
        },
        {
            "pipeline_name": "gold_macro_indicators_ecb",
            "source_name": "ecb",
            "target_table": f"{catalog}.gld_macro.dp_macro_indicators",
            "watermark_column": "observation_date",
            "watermark_type": "observation_date",
            "filter_sql": "source_system = 'ecb'",
        },
        {
            "pipeline_name": "gold_macro_indicators_fred",
            "source_name": "fred",
            "target_table": f"{catalog}.gld_macro.dp_macro_indicators",
            "watermark_column": "observation_date",
            "watermark_type": "observation_date",
            "filter_sql": "source_system = 'fred'",
        },
    ]


def build_dq_specs(catalog: str):
    return [
        {
            "pipeline_name": "silver_market_crypto_ohlc_1d",
            "layer": "silver",
            "source_name": "coinbase",
            "quarantine_table": f"{catalog}.slv_market.crypto_ohlc_1d_quarantine",
            "target_table": f"{catalog}.slv_market.crypto_ohlc_1d",
        },
        {
            "pipeline_name": "silver_macro_ecb_fx_ref_rates_daily",
            "layer": "silver",
            "source_name": "ecb",
            "quarantine_table": f"{catalog}.slv_macro.ecb_fx_ref_rates_daily_quarantine",
            "target_table": f"{catalog}.slv_macro.ecb_fx_ref_rates_daily",
        },
        {
            "pipeline_name": "silver_macro_fred_series_clean",
            "layer": "silver",
            "source_name": "fred",
            "quarantine_table": f"{catalog}.slv_macro.fred_series_clean_quarantine",
            "target_table": f"{catalog}.slv_macro.fred_series_clean",
        },
    ]


def ensure_table_exists(table_name: str):
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table {table_name} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )


def build_metadata_json(payload: dict) -> str:
    return json.dumps(payload, sort_keys=True, default=str)


def upsert_pipeline_run_log(log_table: str, payload: dict):
    log_df = spark.createDataFrame([
        (
            payload["run_id"],
            payload["pipeline_name"],
            payload["layer"],
            payload.get("source_name"),
            payload["target_table"],
            payload["status"],
            payload.get("rows_read"),
            payload.get("rows_written"),
            payload["started_at"],
            payload.get("completed_at"),
            payload.get("error_message"),
            payload.get("metadata_json"),
        )
    ], schema=RUN_LOG_SCHEMA)

    DeltaTable.forName(spark, log_table).alias("tgt").merge(
        log_df.alias("src"),
        "tgt.run_id = src.run_id AND tgt.pipeline_name = src.pipeline_name AND tgt.target_table = src.target_table",
    ).whenMatchedUpdate(
        set={
            "layer": "src.layer",
            "source_name": "src.source_name",
            "status": "src.status",
            "rows_read": "src.rows_read",
            "rows_written": "src.rows_written",
            "started_at": "src.started_at",
            "completed_at": "src.completed_at",
            "error_message": "src.error_message",
            "metadata_json": "src.metadata_json",
        }
    ).whenNotMatchedInsertAll().execute()


def build_ingestion_state_rows(monitored_pipelines: list[dict], observed_at: datetime):
    rows = []
    per_pipeline_row_counts = {}
    pipelines_without_rows = []

    for spec in monitored_pipelines:
        ensure_table_exists(spec["target_table"])
        observed_df = spark.table(spec["target_table"])
        if spec["filter_sql"]:
            observed_df = observed_df.filter(F.expr(spec["filter_sql"]))

        summary = observed_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.max(F.col(spec["watermark_column"])).alias("max_watermark"),
        ).collect()[0]

        row_count = int(summary["row_count"])
        max_watermark = summary["max_watermark"]
        per_pipeline_row_counts[spec["pipeline_name"]] = row_count

        if max_watermark is not None:
            watermark_value = max_watermark.isoformat() if hasattr(max_watermark, "isoformat") else str(max_watermark)
            last_success_at = observed_at
            status = "observed"
        else:
            watermark_value = None
            last_success_at = None
            status = "empty"
            pipelines_without_rows.append(spec["pipeline_name"])

        rows.append((
            spec["pipeline_name"],
            spec["source_name"],
            spec["target_table"],
            watermark_value,
            spec["watermark_type"],
            last_success_at,
            None,
            status,
            observed_at,
        ))

    return rows, per_pipeline_row_counts, pipelines_without_rows


def merge_ingestion_state(state_table: str, state_rows: list[tuple]):
    if not state_rows:
        empty_df = spark.createDataFrame([], schema=INGESTION_STATE_SCHEMA)
        return empty_df, 0

    state_df = spark.createDataFrame(state_rows, schema=INGESTION_STATE_SCHEMA)
    DeltaTable.forName(spark, state_table).alias("tgt").merge(
        state_df.alias("src"),
        "tgt.pipeline_name = src.pipeline_name AND tgt.target_table = src.target_table",
    ).whenMatchedUpdate(
        set={
            "source_name": "src.source_name",
            "watermark_value": "src.watermark_value",
            "watermark_type": "src.watermark_type",
            "last_success_at": "src.last_success_at",
            "last_run_id": "src.last_run_id",
            "status": "src.status",
            "updated_at": "src.updated_at",
        }
    ).whenNotMatchedInsertAll().execute()
    return state_df, len(state_rows)


def build_dq_metrics_df(dq_specs: list[dict], measured_at: datetime):
    metric_dfs = []
    dq_source_rows_read = 0

    for spec in dq_specs:
        ensure_table_exists(spec["quarantine_table"])
        quarantine_df = spark.table(spec["quarantine_table"])
        dq_source_rows_read += quarantine_df.count()

        metric_df = (
            quarantine_df.groupBy("run_id", "dq_reason")
            .count()
            .withColumnRenamed("count", "row_count")
            .withColumn("pipeline_name", F.lit(spec["pipeline_name"]))
            .withColumn("layer", F.lit(spec["layer"]))
            .withColumn("source_name", F.lit(spec["source_name"]))
            .withColumn("target_table", F.lit(spec["target_table"]))
            .withColumn("measured_at", F.lit(measured_at))
            .select(
                "run_id",
                "pipeline_name",
                "layer",
                "source_name",
                "target_table",
                "dq_reason",
                "row_count",
                "measured_at",
            )
        )
        metric_dfs.append(metric_df)

    if not metric_dfs:
        return None, dq_source_rows_read

    return reduce(lambda left, right: left.unionByName(right), metric_dfs), dq_source_rows_read


def merge_dq_metrics(dq_metrics_table: str, dq_metrics_df):
    if dq_metrics_df is None:
        return 0, 0, 0

    rows_ready = dq_metrics_df.count()
    if rows_ready == 0:
        return 0, 0, 0

    existing_key_count = (
        dq_metrics_df.select("run_id", "pipeline_name", "target_table", "dq_reason")
        .join(
            spark.table(dq_metrics_table).select("run_id", "pipeline_name", "target_table", "dq_reason"),
            on=["run_id", "pipeline_name", "target_table", "dq_reason"],
            how="inner",
        )
        .count()
    )

    DeltaTable.forName(spark, dq_metrics_table).alias("tgt").merge(
        dq_metrics_df.alias("src"),
        "tgt.run_id = src.run_id AND tgt.pipeline_name = src.pipeline_name AND tgt.target_table = src.target_table AND tgt.dq_reason = src.dq_reason",
    ).whenMatchedUpdate(
        set={
            "layer": "src.layer",
            "source_name": "src.source_name",
            "row_count": "src.row_count",
            "measured_at": "src.measured_at",
        }
    ).whenNotMatchedInsertAll().execute()

    return rows_ready, existing_key_count, rows_ready - existing_key_count


spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
catalog = dbutils.widgets.get("catalog").strip() or "market_macro"
run_id = dbutils.widgets.get("run_id").strip()
started_at = datetime.now(UTC)

state_table = f"{catalog}.obs.obs_ingestion_state"
run_log_table = f"{catalog}.obs.obs_pipeline_run_log"
dq_metrics_table = f"{catalog}.obs.obs_dq_metrics"

ensure_table_exists(state_table)
ensure_table_exists(run_log_table)
ensure_table_exists(dq_metrics_table)

monitored_pipelines = build_monitored_pipelines(catalog)
dq_specs = build_dq_specs(catalog)

pipeline_name = "obs_pipeline_observability_metrics"
target_table = dq_metrics_table

upsert_pipeline_run_log(
    run_log_table,
    {
        "run_id": run_id,
        "pipeline_name": pipeline_name,
        "layer": "obs",
        "source_name": "system",
        "target_table": target_table,
        "status": "started",
        "rows_read": None,
        "rows_written": None,
        "started_at": started_at,
        "completed_at": None,
        "error_message": None,
        "metadata_json": build_metadata_json(
            {
                "state_table": state_table,
                "dq_metrics_table": dq_metrics_table,
                "monitored_pipeline_count": len(monitored_pipelines),
                "dq_source_count": len(dq_specs),
            }
        ),
    },
)

try:
    observed_at = datetime.now(UTC)
    state_rows, per_pipeline_observed_rows, pipelines_without_rows = build_ingestion_state_rows(
        monitored_pipelines,
        observed_at,
    )
    state_df, state_rows_upserted = merge_ingestion_state(state_table, state_rows)
    dq_metrics_df, dq_source_rows_read = build_dq_metrics_df(dq_specs, observed_at)
    dq_rows_ready, dq_rows_to_update, dq_rows_to_insert = merge_dq_metrics(dq_metrics_table, dq_metrics_df)
    dq_rows_merged = dq_rows_ready

    total_observed_rows = sum(per_pipeline_observed_rows.values())

    result = {
        "status": "success",
        "pipeline_name": pipeline_name,
        "catalog": catalog,
        "run_id": run_id,
        "state_table": state_table,
        "dq_metrics_table": dq_metrics_table,
        "run_log_table": run_log_table,
        "pipelines_observed": len(monitored_pipelines),
        "state_rows_upserted": state_rows_upserted,
        "dq_source_tables": len(dq_specs),
        "dq_source_rows_read": dq_source_rows_read,
        "dq_rows_ready": dq_rows_ready,
        "dq_rows_to_update": dq_rows_to_update,
        "dq_rows_to_insert": dq_rows_to_insert,
        "dq_rows_merged": dq_rows_merged,
        "total_observed_rows": total_observed_rows,
        "pipelines_without_rows": pipelines_without_rows,
        "per_pipeline_observed_rows": per_pipeline_observed_rows,
    }

    completed_at = datetime.now(UTC)
    upsert_pipeline_run_log(
        run_log_table,
        {
            "run_id": run_id,
            "pipeline_name": pipeline_name,
            "layer": "obs",
            "source_name": "system",
            "target_table": target_table,
            "status": "success",
            "rows_read": total_observed_rows + dq_source_rows_read,
            "rows_written": state_rows_upserted + dq_rows_merged,
            "started_at": started_at,
            "completed_at": completed_at,
            "error_message": None,
            "metadata_json": build_metadata_json(result),
        },
    )

    display(state_df.orderBy("source_name", "pipeline_name"))
    if dq_rows_ready > 0:
        display(dq_metrics_df.orderBy("pipeline_name", "run_id", "dq_reason"))
    recent_runs_df = spark.table(run_log_table).orderBy(F.col("started_at").desc()).limit(20)
    display(recent_runs_df)
except Exception as exc:
    completed_at = datetime.now(UTC)
    upsert_pipeline_run_log(
        run_log_table,
        {
            "run_id": run_id,
            "pipeline_name": pipeline_name,
            "layer": "obs",
            "source_name": "system",
            "target_table": target_table,
            "status": "failed",
            "rows_read": None,
            "rows_written": None,
            "started_at": started_at,
            "completed_at": completed_at,
            "error_message": str(exc),
            "metadata_json": build_metadata_json(
                {
                    "state_table": state_table,
                    "dq_metrics_table": dq_metrics_table,
                    "monitored_pipeline_count": len(monitored_pipelines),
                    "dq_source_count": len(dq_specs),
                }
            ),
        },
    )
    raise

result
